# Module 08, Unit 02 — Variational Inference: Derivation & Implementation

> **Threat Surfaces: Multivariable Calculus for AI Security**
> fischer³ Education | Module 08 | Unit 02

| | |
|---|---|
| **Estimated time** | 70–80 min |
| **Exercises** | Download PDF from the course repository |
| **Statistical threads** | `[BAY]` · `[MLE]` · `[GLM]` |
| **Prerequisites** | Module 08 Unit 01 — complete the derivation before running code |

---

## Learning Objectives

By the end of this unit you will be able to:

- [ ] State the reparameterization trick and explain why it enables gradient-based ELBO optimization
- [ ] Derive $\nabla_{\boldsymbol{m}}\,\text{ELBO}$ and $\nabla_{\boldsymbol{\rho}}\,\text{ELBO}$ via the reparameterization
- [ ] Implement the ELBO and its gradients from scratch in NumPy
- [ ] Run gradient ascent on the ELBO and monitor convergence
- [ ] Explain the role of the Monte Carlo sample size $K$ in bias and variance of the gradient estimates

---

## Recap from Unit 01

We are maximizing:

$$
\text{ELBO}(\boldsymbol{m}, \boldsymbol{\rho}) = \underbrace{\mathbb{E}_{q}\!\left[\log p(\boldsymbol{y} \mid \boldsymbol{X}, \boldsymbol{w})\right]}_{\text{expected log-likelihood — needs MC}} - \underbrace{D_{\text{KL}}(q \| p)}_{\text{closed form}}
$$

where $q(\boldsymbol{w}) = \mathcal{N}(\boldsymbol{m}, \text{diag}(e^{2\boldsymbol{\rho}}))$. The KL term is known analytically. The expected log-likelihood requires Monte Carlo. The outstanding problem: how do we differentiate through the sampling step?

---

## 1. The Reparameterization Trick

### 1.1 The Problem

We want $\nabla_{\boldsymbol{m}}\,\mathbb{E}_{q}[\log p(\boldsymbol{y}|\boldsymbol{X},\boldsymbol{w})]$. Written out:

$$
\nabla_{\boldsymbol{m}}\,\mathbb{E}_{q}[\log p] = \nabla_{\boldsymbol{m}}\int \log p(\boldsymbol{y}|\boldsymbol{X},\boldsymbol{w})\,\mathcal{N}(\boldsymbol{w};\boldsymbol{m}, \text{diag}(e^{2\boldsymbol{\rho}}))\,d\boldsymbol{w}
$$

The problem: $\boldsymbol{m}$ appears in the distribution we are integrating over, not in the integrand. We cannot swap gradient and integral naively because the distribution depends on the parameter.

### 1.2 The Solution

**Reparameterize** the sample $\boldsymbol{w} \sim q$ as a deterministic function of $\boldsymbol{m}$, $\boldsymbol{\rho}$, and a noise variable $\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I})$:

$$
\boldsymbol{w} = \boldsymbol{m} + e^{\boldsymbol{\rho}} \odot \boldsymbol{\varepsilon}, \qquad \boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0}, \boldsymbol{I})
$$

where $\odot$ is elementwise multiplication and $e^{\boldsymbol{\rho}} = (e^{\rho_0}, e^{\rho_1}, e^{\rho_2})^{\top} = \boldsymbol{s}$.

Now the expectation over $q$ becomes an expectation over $\boldsymbol{\varepsilon} \sim \mathcal{N}(\boldsymbol{0},\boldsymbol{I})$:

$$
\mathbb{E}_{q}[\log p(\boldsymbol{y}|\boldsymbol{X},\boldsymbol{w})] = \mathbb{E}_{\boldsymbol{\varepsilon}}[\log p(\boldsymbol{y}|\boldsymbol{X},\, \boldsymbol{m} + e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon})]
$$

The distribution of $\boldsymbol{\varepsilon}$ no longer depends on $\boldsymbol{m}$ or $\boldsymbol{\rho}$. We can now **swap gradient and expectation**:

$$
\nabla_{\boldsymbol{m}}\,\mathbb{E}_{\boldsymbol{\varepsilon}}[\log p(\boldsymbol{y}|\boldsymbol{X}, \boldsymbol{m}+e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon})] = \mathbb{E}_{\boldsymbol{\varepsilon}}\!\left[\nabla_{\boldsymbol{m}}\,\log p(\boldsymbol{y}|\boldsymbol{X}, \boldsymbol{m}+e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon})\right]
$$

### 1.3 The Chain Rule Applied

Let $\boldsymbol{w} = \boldsymbol{m} + e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon}$ (this is a function of $\boldsymbol{m}$ and $\boldsymbol{\rho}$). By the chain rule (Module 03 Unit 03):

$$
\nabla_{\boldsymbol{m}}\,\log p(\boldsymbol{y}|\boldsymbol{X},\boldsymbol{w}) = \frac{\partial \boldsymbol{w}}{\partial \boldsymbol{m}}^{\top} \nabla_{\boldsymbol{w}}\,\log p(\boldsymbol{y}|\boldsymbol{X},\boldsymbol{w}) = \boldsymbol{I} \cdot \nabla_{\boldsymbol{w}}\,\log p
$$

since $\frac{\partial w_j}{\partial m_j} = 1$ and $\frac{\partial w_j}{\partial m_k} = 0$ for $k \neq j$.

$$
\nabla_{\boldsymbol{\rho}}\,\log p(\boldsymbol{y}|\boldsymbol{X},\boldsymbol{w}) = \frac{\partial \boldsymbol{w}}{\partial \boldsymbol{\rho}}^{\top} \nabla_{\boldsymbol{w}}\,\log p = \text{diag}(e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon})\,\nabla_{\boldsymbol{w}}\,\log p = (e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon})\odot\nabla_{\boldsymbol{w}}\,\log p
$$

since $\frac{\partial w_j}{\partial \rho_j} = e^{\rho_j}\varepsilon_j$.

---

## 2. The ELBO Gradients

### 2.1 Gradient of the Expected Log-Likelihood

From the logistic regression gradient (Module 03 Unit 02, Section 4.3):

$$
\nabla_{\boldsymbol{w}}\,\log p(\boldsymbol{y}|\boldsymbol{X},\boldsymbol{w}) = \boldsymbol{X}^{\top}(\boldsymbol{y} - \sigma(\boldsymbol{X}\boldsymbol{w}))
$$

Combining with the reparameterization:

$$
\nabla_{\boldsymbol{m}}\,\mathbb{E}_q[\log p] \approx \frac{1}{K}\sum_{k=1}^K \boldsymbol{X}^{\top}(\boldsymbol{y} - \sigma(\boldsymbol{X}\boldsymbol{w}^{(k)}))
$$

$$
\nabla_{\boldsymbol{\rho}}\,\mathbb{E}_q[\log p] \approx \frac{1}{K}\sum_{k=1}^K (e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon}^{(k)})\odot\boldsymbol{X}^{\top}(\boldsymbol{y} - \sigma(\boldsymbol{X}\boldsymbol{w}^{(k)}))
$$

where $\boldsymbol{w}^{(k)} = \boldsymbol{m} + e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon}^{(k)}$ and $\boldsymbol{\varepsilon}^{(k)} \overset{\text{iid}}{\sim} \mathcal{N}(\boldsymbol{0},\boldsymbol{I})$.

### 2.2 Gradient of the KL Term

The KL term is fully analytical (Section 7 of Unit 01):

$$
D_{\text{KL}} = \frac{1}{2}\sum_{j}\left[\frac{e^{2\rho_j}}{\sigma_0^2} + \frac{m_j^2}{\sigma_0^2} - 1 - 2\rho_j + \log\sigma_0^2\right]
$$

Differentiating component by component:

$$
\frac{\partial D_{\text{KL}}}{\partial m_j} = \frac{m_j}{\sigma_0^2}
$$

$$
\frac{\partial D_{\text{KL}}}{\partial \rho_j} = \frac{e^{2\rho_j}}{\sigma_0^2} - 1
$$

In vector form: $\nabla_{\boldsymbol{m}}\,D_{\text{KL}} = \boldsymbol{m}/\sigma_0^2$ and $\nabla_{\boldsymbol{\rho}}\,D_{\text{KL}} = e^{2\boldsymbol{\rho}}/\sigma_0^2 - \boldsymbol{1}$.

### 2.3 Full ELBO Gradient

Since ELBO $= \mathbb{E}_q[\log p] - D_{\text{KL}}$, the gradient is:

$$
\boxed{\nabla_{\boldsymbol{m}}\,\text{ELBO} \approx \frac{1}{K}\sum_{k=1}^K \boldsymbol{X}^{\top}(\boldsymbol{y} - \hat{\boldsymbol{p}}^{(k)}) - \frac{\boldsymbol{m}}{\sigma_0^2}}
$$

$$
\boxed{\nabla_{\boldsymbol{\rho}}\,\text{ELBO} \approx \frac{1}{K}\sum_{k=1}^K \left[(e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon}^{(k)})\odot\boldsymbol{X}^{\top}(\boldsymbol{y} - \hat{\boldsymbol{p}}^{(k)})\right] - \left(\frac{e^{2\boldsymbol{\rho}}}{\sigma_0^2} - \boldsymbol{1}\right)}
$$

where $\hat{\boldsymbol{p}}^{(k)} = \sigma(\boldsymbol{X}\boldsymbol{w}^{(k)})$.

**Reading the $\boldsymbol{m}$ gradient:**
- $\frac{1}{K}\sum_k \boldsymbol{X}^{\top}(\boldsymbol{y} - \hat{\boldsymbol{p}}^{(k)})$: average gradient of the log-likelihood over sampled weights — this is the expected score, pushing $\boldsymbol{m}$ toward better-fitting weight values
- $-\boldsymbol{m}/\sigma_0^2$: the KL gradient — this is $-\nabla_{\boldsymbol{m}}(\text{Ridge penalty}/2)$, pulling $\boldsymbol{m}$ back toward zero

The ELBO gradient for $\boldsymbol{m}$ is exactly the gradient of logistic regression with Ridge penalty, evaluated at a noisy weight $\boldsymbol{w}^{(k)}$ rather than at $\boldsymbol{m}$ itself.

---

## 3. The Gradient Ascent Algorithm

With the gradients derived, the optimization is straightforward gradient ascent:

$$
\boldsymbol{m}^{(t+1)} = \boldsymbol{m}^{(t)} + \alpha\,\nabla_{\boldsymbol{m}}\,\text{ELBO}(\boldsymbol{m}^{(t)}, \boldsymbol{\rho}^{(t)})
$$

$$
\boldsymbol{\rho}^{(t+1)} = \boldsymbol{\rho}^{(t)} + \alpha\,\nabla_{\boldsymbol{\rho}}\,\text{ELBO}(\boldsymbol{m}^{(t)}, \boldsymbol{\rho}^{(t)})
$$

At each step we draw $K$ fresh noise samples $\boldsymbol{\varepsilon}^{(1)},\ldots,\boldsymbol{\varepsilon}^{(K)} \overset{\text{iid}}{\sim} \mathcal{N}(\boldsymbol{0},\boldsymbol{I})$ and compute the stochastic ELBO estimate and its gradients.

**Choice of $K$:**
- $K = 1$: stochastic, high variance, fast per step — common in practice
- $K = 10$: lower variance, more stable — used in this implementation
- $K \to \infty$: deterministic (the true expected value), no longer needed with reparameterization since the KL is exact

---

## Python: Implementation from Scratch

Every function below is a direct transcription of the mathematical derivation above. Read each function against its corresponding equation before running.

In [ ]:
import subprocess, sys
for pkg in ["numpy", "matplotlib", "scipy", "scikit-learn"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
print("All packages installed.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (9,6), 'font.size': 11,
                     'axes.titlesize': 13, 'lines.linewidth': 2, 'figure.dpi': 120})
TS_BLUE='#1e64b4'; TS_AMBER='#c87814'; TS_GREEN='#1e8c50'
TS_GRAY='#64646e'; TS_RED='#b41e1e'; TS_LIGHT='#f5f7fa'

# ── Reproduce dataset from Unit 01 ──────────────────────────
iris = load_iris()
mask = iris.target < 2
X_raw = iris.data[mask, :2]
y = iris.target[mask].astype(float)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
n = len(y)
X = np.column_stack([np.ones(n), X_scaled])   # (100, 3)
p_dim = X.shape[1]

print('Data loaded. X shape:', X.shape, ' y shape:', y.shape)

### Section 1 — Core Functions

In [ ]:
# ============================================================
# Core functions — each maps exactly to a derivation in the notes
# ============================================================

def sigmoid(z):
    """σ(z) = 1/(1+e^{-z}), clipped for numerical stability."""
    return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

def log_likelihood(w, X, y):
    """
    Log-likelihood: Σᵢ [yᵢ log σ(wᵀxᵢ) + (1-yᵢ) log(1-σ(wᵀxᵢ))]
    Gradient: Xᵀ(y - σ(Xw))   [Module 03 Unit 02]
    """
    p_hat = sigmoid(X @ w)
    p_hat = np.clip(p_hat, 1e-12, 1-1e-12)
    return np.sum(y * np.log(p_hat) + (1-y) * np.log(1-p_hat))

def grad_log_likelihood(w, X, y):
    """∇_w log p(y|X,w) = Xᵀ(y - σ(Xw))   [Module 03 Unit 02]"""
    return X.T @ (y - sigmoid(X @ w))

def kl_divergence(m, log_s, sigma0_sq):
    """
    KL(N(m, diag(exp(2*log_s))) || N(0, σ₀²I))
    = (1/2) Σⱼ [sⱼ²/σ₀² + mⱼ²/σ₀² - 1 - log(sⱼ²/σ₀²)]
    [Unit 01 Section 7]
    """
    s_sq = np.exp(2 * log_s)
    return 0.5 * np.sum(s_sq/sigma0_sq + m**2/sigma0_sq - 1 - np.log(s_sq/sigma0_sq))

def grad_kl(m, log_s, sigma0_sq):
    """
    ∂KL/∂mⱼ = mⱼ/σ₀²
    ∂KL/∂ρⱼ = exp(2ρⱼ)/σ₀² - 1    [Unit 02 Section 2.2]
    """
    s_sq = np.exp(2 * log_s)
    g_m   = m / sigma0_sq
    g_rho = s_sq / sigma0_sq - 1.0
    return g_m, g_rho

def elbo_and_grad(m, log_s, X, y, sigma0_sq, K=10, rng=None):
    """
    Compute ELBO estimate and its gradients via reparameterization.
    w^(k) = m + exp(log_s) ⊙ ε^(k),  ε^(k) ~ N(0,I)

    Returns: (elbo_estimate, grad_m, grad_rho)
    """
    if rng is None:
        rng = np.random.default_rng()
    s = np.exp(log_s)                          # (p,)

    # Draw K noise samples
    eps = rng.standard_normal((K, len(m)))     # (K, p)

    # Reparameterized samples: w^(k) = m + s ⊙ ε^(k)
    w_samples = m + s * eps                    # (K, p)

    # ── Expected log-likelihood and its gradients ─────────────
    ll_vals  = np.array([log_likelihood(w_samples[k], X, y) for k in range(K)])
    g_ll_w   = np.array([grad_log_likelihood(w_samples[k], X, y) for k in range(K)])  # (K,p)

    # ∇_m E_q[log p] ≈ (1/K) Σ_k ∇_w log p(w^(k))     [Section 2.1]
    g_ll_m   = g_ll_w.mean(axis=0)                                       # (p,)

    # ∇_ρ E_q[log p] ≈ (1/K) Σ_k (s ⊙ ε^(k)) ⊙ ∇_w log p(w^(k))
    g_ll_rho = (s * eps * g_ll_w).mean(axis=0)                           # (p,)

    # ── KL term (analytical) ──────────────────────────────────
    kl       = kl_divergence(m, log_s, sigma0_sq)
    g_kl_m, g_kl_rho = grad_kl(m, log_s, sigma0_sq)

    # ── ELBO = E_q[log p] - KL ────────────────────────────────
    elbo = ll_vals.mean() - kl

    # ∇ ELBO = ∇ E_q[log p] - ∇ KL
    grad_m   = g_ll_m   - g_kl_m
    grad_rho = g_ll_rho - g_kl_rho

    return elbo, grad_m, grad_rho

print('All core functions defined.')

# ── Quick sanity check at zero initialization ────────────────
rng0 = np.random.default_rng(0)
m0     = np.zeros(p_dim)
log_s0 = np.zeros(p_dim)   # s = 1 initially
sigma0_sq = 1.0

elbo0, gm0, gr0 = elbo_and_grad(m0, log_s0, X, y, sigma0_sq, K=50, rng=rng0)
print(f'\nAt initialization (m=0, log_s=0):')
print(f'  ELBO    = {elbo0:.4f}')
print(f'  ∇_m     = {np.round(gm0,4)}')
print(f'  ∇_ρ     = {np.round(gr0,4)}')
print(f'  KL      = {kl_divergence(m0, log_s0, sigma0_sq):.4f}  (expect 0 at m=0, s=1, σ₀=1)')

**What do you see?** At initialization $\boldsymbol{m}=\boldsymbol{0}$, $\boldsymbol{\rho}=\boldsymbol{0}$ (i.e., $\boldsymbol{s}=\boldsymbol{1}$), with $\sigma_0^2=1$: the KL is zero — $q = p$ exactly, so there is no divergence. The ELBO equals the expected log-likelihood under a random Gaussian prior. The gradients are nonzero, pointing toward higher-likelihood weight configurations.

### Section 2 — Gradient Ascent on the ELBO

In [ ]:
# ============================================================
# Gradient ascent on the ELBO
# ============================================================
def run_vi(X, y, sigma0_sq=1.0, lr=0.05, n_iter=500, K=10, seed=42):
    """Full variational inference loop via gradient ascent on ELBO."""
    rng = np.random.default_rng(seed)
    p   = X.shape[1]

    # Initialize
    m     = np.zeros(p)
    log_s = np.zeros(p)   # s = 1

    elbo_history  = []
    m_history     = [m.copy()]
    log_s_history = [log_s.copy()]

    for t in range(n_iter):
        elbo, gm, gr = elbo_and_grad(m, log_s, X, y, sigma0_sq, K=K, rng=rng)
        m     = m     + lr * gm
        log_s = log_s + lr * gr
        elbo_history.append(elbo)
        m_history.append(m.copy())
        log_s_history.append(log_s.copy())

    return {
        'm':          m,
        'log_s':      log_s,
        's':          np.exp(log_s),
        'elbo':       np.array(elbo_history),
        'm_hist':     np.array(m_history),
        'log_s_hist': np.array(log_s_history),
    }

# Run with default hyperparameters
results = run_vi(X, y, sigma0_sq=1.0, lr=0.05, n_iter=600, K=10)

print('=== Variational Inference Converged ===')
print(f'Posterior mean m̂     = {np.round(results["m"], 4)}')
print(f'Posterior std  ŝ     = {np.round(results["s"], 4)}')
print(f'Final ELBO           = {results["elbo"][-1]:.4f}')
print(f'KL at convergence    = {kl_divergence(results["m"], results["log_s"], 1.0):.4f}')

# ── Convergence plots ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Left: ELBO history
ax = axes[0]
ax.plot(results['elbo'], color=TS_BLUE, lw=2)
ax.axhline(results['elbo'][-1], color=TS_AMBER, lw=1.5, ls='--',
           label=f'Final ELBO = {results["elbo"][-1]:.2f}')
ax.set_xlabel('Iteration'); ax.set_ylabel('ELBO')
ax.set_title('ELBO convergence')
ax.legend(fontsize=9)

# Middle: mean trajectory
ax2 = axes[1]
labels_m = ['$m_0$ (bias)', '$m_1$ (sepal L)', '$m_2$ (sepal W)']
colors_m = [TS_BLUE, TS_AMBER, TS_GREEN]
for j, (lab, col) in enumerate(zip(labels_m, colors_m)):
    ax2.plot(results['m_hist'][:, j], color=col, lw=2, label=lab)
ax2.axhline(0, color=TS_GRAY, lw=0.8)
ax2.set_xlabel('Iteration'); ax2.set_ylabel('Value')
ax2.set_title('Posterior mean $m$ trajectory')
ax2.legend(fontsize=9)

# Right: log_s trajectory (standard deviation)
ax3 = axes[2]
labels_s = ['$s_0$ (bias)', '$s_1$ (sepal L)', '$s_2$ (sepal W)']
for j, (lab, col) in enumerate(zip(labels_s, colors_m)):
    ax3.plot(np.exp(results['log_s_hist'][:, j]), color=col, lw=2, label=lab)
ax3.set_xlabel('Iteration'); ax3.set_ylabel('Value')
ax3.set_title('Posterior std $s = e^\\rho$ trajectory')
ax3.legend(fontsize=9)

plt.suptitle('Variational inference: gradient ascent on ELBO', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**What do you see?**

- **ELBO**: increases monotonically (with some stochastic noise from the $K=10$ Monte Carlo estimate) and converges to a stable value. The ELBO is not guaranteed to increase at every step because the gradient is stochastic — a smoother curve requires larger $K$ or a smaller learning rate.
- **Posterior mean** $\boldsymbol{m}$: starts at zero and moves toward the MAP/MLE direction. $m_1$ (sepal length) and $m_2$ (sepal width) converge to their optimal values; the sign indicates which direction in feature space separates the classes.
- **Posterior std** $\boldsymbol{s}$: starts at 1 (the prior standard deviation) and shrinks as the data informs the posterior. Parameters with more information (stronger signal) get smaller posterior uncertainty.

### Section 3 — Effect of $K$ and Learning Rate

In [ ]:
# ============================================================
# Effect of Monte Carlo sample size K and learning rate
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: different K values
ax = axes[0]
for K_val, col in [(1, TS_RED), (5, TS_AMBER), (20, TS_GREEN), (50, TS_BLUE)]:
    res_k = run_vi(X, y, sigma0_sq=1.0, lr=0.05, n_iter=400, K=K_val, seed=7)
    ax.plot(res_k['elbo'], color=col, lw=1.8, alpha=0.85,
            label=f'K={K_val}')
ax.set_xlabel('Iteration'); ax.set_ylabel('ELBO')
ax.set_title('Effect of Monte Carlo sample size $K$\n(more K → smoother but slower per step)')
ax.legend(fontsize=9)

# Right: different learning rates
ax2 = axes[1]
for lr_val, col in [(0.005, TS_RED), (0.02, TS_AMBER), (0.05, TS_BLUE), (0.15, TS_GREEN)]:
    res_lr = run_vi(X, y, sigma0_sq=1.0, lr=lr_val, n_iter=600, K=10, seed=7)
    ax2.plot(res_lr['elbo'], color=col, lw=1.8, alpha=0.85,
             label=f'α={lr_val}')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('ELBO')
ax2.set_title('Effect of learning rate $\\alpha$')
ax2.legend(fontsize=9)

plt.suptitle('Hyperparameter sensitivity of ELBO optimization',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

**What do you see?** The variance-bias trade-off in $K$ and the convergence speed trade-off in $\alpha$ match the theoretical predictions from Module 04 Unit 02. Large $K$ reduces gradient variance (smoother ELBO) but each iteration costs $K$ forward passes. Large $\alpha$ converges faster initially but may overshoot or oscillate. These are the same trade-offs we saw for gradient descent in Module 03 Unit 02 — they apply identically here because the ELBO optimization is gradient ascent on a different objective function but with the same algorithmic structure.

In [ ]:
# Extension workspace
# 1. Gradient check: verify ∇_m ELBO numerically via finite differences.
#    Perturb m_j by ε, compute ELBO, and estimate ∂ELBO/∂m_j ≈ (ELBO(m+εeⱼ)-ELBO(m-εeⱼ))/(2ε).
#    Compare to the analytical gradient. Use K=1000 to reduce MC noise.
#
# 2. Adam optimizer: implement the Adam update rule for (m, log_s).
#    Adam maintains per-parameter first and second moment estimates.
#    Does it converge faster or more stably than plain gradient ascent?
#
# 3. Full-covariance variational family: replace diag(s²) with a full
#    lower-triangular Cholesky factor L, so Σ = LLᵀ.
#    How does the reparameterization change? (w = m + L ε)
#    What is the new KL formula? (Module 06 Unit 03 closed form applies.)


---

## Summary

| Concept | Expression | Module origin |
|---|---|---|
| Reparameterization | $\boldsymbol{w} = \boldsymbol{m} + e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon}$, $\boldsymbol{\varepsilon}\sim\mathcal{N}(\boldsymbol{0},\boldsymbol{I})$ | Module 07 Unit 01 (score trick) |
| ELBO gradient $\boldsymbol{m}$ | $\frac{1}{K}\sum_k \boldsymbol{X}^{\top}(\boldsymbol{y}-\hat{\boldsymbol{p}}^{(k)}) - \boldsymbol{m}/\sigma_0^2$ | Module 03 Unit 02 + this unit |
| ELBO gradient $\boldsymbol{\rho}$ | $\frac{1}{K}\sum_k(e^{\boldsymbol{\rho}}\odot\boldsymbol{\varepsilon}^{(k)})\odot\boldsymbol{X}^{\top}(\boldsymbol{y}-\hat{\boldsymbol{p}}^{(k)}) - (e^{2\boldsymbol{\rho}}/\sigma_0^2 - \boldsymbol{1})$ | Module 03 Unit 03 (chain rule) |
| Gradient ascent | $\boldsymbol{m} \leftarrow \boldsymbol{m} + \alpha\nabla_{\boldsymbol{m}}\text{ELBO}$ | Module 03 Unit 02 |
| KL gradient $\boldsymbol{m}$ | $\boldsymbol{m}/\sigma_0^2$ — Ridge penalty gradient | Module 04 Unit 02 |
| KL gradient $\boldsymbol{\rho}$ | $e^{2\boldsymbol{\rho}}/\sigma_0^2 - \boldsymbol{1}$ | Module 07 Unit 03 |

**Up next:** Unit 03 — Posterior Analysis & Course Retrospective

---
*Module 08, Unit 02 — Threat Surfaces | fischer³ Education*